# Chapter 7 — How Many Dimensions Does Meaning Need?

**Book alignment:** Embeddings From First Principles, Chapter 7

**Question this notebook isolates:** Nominal dimension is a spec, not a measurement. Does a
768-d RELATE space have an effective rank of a few hundred and an intrinsic dimension in
the single digits — and is the safe truncation dimension *task-specific* (clustering
survives d = 8 where retrieval needs d ≈ 24), with the knee *not equal* to any single
geometric quantity?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave: str, name: str) -> dict:
    return json.loads((EXP / wave / "artifacts" / name).read_text())

## 1. Nominal vs effective rank — reproduce the gap

In [ ]:
n, d, true_rank = 500, 128, 12
A = rng.standard_normal((n, true_rank)) @ rng.standard_normal((true_rank, d))
A = A + rng.standard_normal((n, d)) * 0.05          # noise fills every direction

S = np.linalg.svd(A, compute_uv=False)
p = S ** 2 / (S ** 2).sum()
eff_rank = float(np.exp(-(p * np.log(p)).sum()))    # spectral entropy
print(f"nominal dimension : {d}")
print(f"matrix rank       : {np.linalg.matrix_rank(A)}   (noise -> full)")
print(f"effective rank    : {eff_rank:.1f}   (~ the {true_rank} directions that carry variance)")
assert np.linalg.matrix_rank(A) == d and eff_rank < 3 * true_rank

## 2. RELATE's real dimensionality (Wave 2)

In [ ]:
dim = art("wave2", "dimensionality-report.json")["models"]
for m, v in dim.items():
    print(f"  {m:12} nominal {v['embedding_dim']:>4}  eff.rank {v['effective_rank']:>5.0f}"
          f"  95%-var {v['var95_dim']:>4}  TwoNN-ID {v['twonn_id']:.1f}  MLE-ID {v['mle_id_k10']:.1f}")
mp = dim["mpnet-base"]
assert mp["twonn_id"] < 10 < mp["effective_rank"] < mp["embedding_dim"]
print("\nfive different 'sizes' spanning two orders of magnitude; the ID estimators are single digits")

## 3. The safe truncation dimension is set by the task, not the spectrum (Wave 2)

In [ ]:
rc = art("wave2", "retention-curves.json")
dims, knees = rc["dims"], rc["knees"]
print("d kept :", dims)
for task in ("retrieval", "sts", "clustering", "hard_negative"):
    print(f"  {task:13}", [round(v, 3) for v in rc["curves"][task]], "  knee(tol5) =", knees[task]["tol5"])

# clustering + hard-negative survive d=8; retrieval needs ~24; sts ~32
assert knees["clustering"]["tol5"] == 8 and knees["retrieval"]["tol5"] == 24
assert rc["knee_spread_tol5"] == 24
# the knee is NOT the intrinsic dimension
assert knees["retrieval"]["tol5"] != round(dim["mpnet-base"]["twonn_id"])
# clustering ARI actually RISES under compression (the extra dims are noise for k-means)
assert rc["curves"]["clustering"][0] > rc["curves"]["clustering"][-1]
print("\ngeometry can suggest redundancy; only a per-task retention curve can authorize compression")

## What we earned

A 768-number vector uses a few hundred directions (effective rank) and traces a manifold of
intrinsic dimension ~5. *Some* compression is always safe — but *how much* is task-specific:
on RELATE, clustering and hard-negative discrimination survive d = 8 while retrieval needs
d ≈ 24 and the STS proxy d ≈ 32, and none of those knees equals the intrinsic-dimension
estimate. Truncation needs a measured retention curve per task, not a spectrum ratio.

**Notebook 08 / Chapter 8** measures the *shape* of how a space uses its directions —
anisotropy, dominant directions — and when the standard whitening repair helps vs hurts.